# Svara TTS API Demo

This notebook demonstrates the Intel Text-to-Speech API with 88 multi-language voices.

## System Status
- ✅ TTS API Pod: snac-decoder (Kubernetes)
- ✅ Service Endpoint: NodePort (External usage)
- ✅ VLLM Backend: Connected
- ✅ SNAC Codec: Loaded
- ✅ Available Voices: 88

## Setup
Import required libraries

In [14]:
import requests
import json
from IPython.display import Audio, display
import pandas as pd
import os

def get_service_endpoint(service_name="snac-decoder-service", namespace="default", port=30800, node_ip="172.31.51.24"):
    """
    Get the endpoint for a Kubernetes service.
    
    Priority:
    1. Environment variable TTS_API_URL (for manual override)
    2. NodePort Access (External Access via Node IP)
    3. Kubernetes DNS (automatic, no API calls needed)
    """
    # 1. Check environment variable first
    if os.environ.get("TTS_API_URL"):
        return os.environ.get("TTS_API_URL")
    
    # 2. Try NodePort Access (External)
    # For NodePort, we use the Node's IP (or localhost if port-forwarded/local) and the NodePort
    nodeport_endpoint = f"http://{node_ip}:{port}"
    
    try:
        response = requests.get(f"{nodeport_endpoint}/health", timeout=2)
        if response.status_code == 200:
            return nodeport_endpoint
    except:
        pass
    
    # 3. Fallback to K8s DNS (Inside Cluster)
    dns_endpoint = f"http://{service_name}.{namespace}.svc.cluster.local:8000"
    return dns_endpoint

# TTS API Configuration
# Update node_ip to your actual K8s node IP if not running on the same machine
# We use the NodePort 30800 and the Node IP discovered from the cluster
BASE_URL = get_service_endpoint(port=8000, node_ip="35.86.73.253") 

print(f"✅ Connected to TTS API at: {BASE_URL}")

✅ Connected to TTS API at: http://snac-decoder-service.default.svc.cluster.local:8000


## 1. Health Check
Verify the TTS API server is healthy and connected to all services

In [22]:
try:
    response = requests.get(f"{BASE_URL}/health", timeout=5)
    health = response.json()
    
    print("🏥 Health Check Results:")
    print(f"  Status: {health['status'].upper()}")
    print(f"  SNAC Codec: {'✅ Loaded' if health['snac_loaded'] else '❌ Not Loaded'}")
    print(f"  VLLM Backend: {'✅ Connected' if health['vllm_connected'] else '❌ Disconnected'}")
    print(f"  Available Voices: {health['voices_loaded']}")
    
except Exception as e:
    print(f"❌ Health check failed: {e}")

🏥 Health Check Results:
  Status: HEALTHY
  SNAC Codec: ✅ Loaded
  VLLM Backend: ✅ Connected
  Available Voices: 88


## 2. List Available Voices
Explore the 88 voices across multiple languages

In [16]:
try:
    response = requests.get(f"{BASE_URL}/v1/voices")
    voices = response.json()
    
    print(f"📢 Found {len(voices)} voices\n")
    
    # Create a DataFrame for better visualization
    df = pd.DataFrame(voices)
    
    # Show language distribution
    print("🌍 Languages Available:")
    print(df['language'].value_counts().head(10))
    
    print("\n📝 Sample Voices:")
    display(df[['id', 'name', 'language', 'gender']].head(15))
    
except Exception as e:
    print(f"❌ Failed to list voices: {e}")

📢 Found 88 voices

🌍 Languages Available:
language
hi     52
bn      2
mr      2
te      2
kn      2
bh      2
mag     2
hne     2
mai     2
as      2
Name: count, dtype: int64

📝 Sample Voices:


,id,name,language,gender
0,hi_male,Hindi (Male),hi,male
1,hi_female,Hindi (Female),hi,female
2,bn_male,Bengali (Male),bn,male
3,bn_female,Bengali (Female),bn,female
4,mr_male,Marathi (Male),mr,male
5,mr_female,Marathi (Female),mr,female
6,te_male,Telugu (Male),te,male
7,te_female,Telugu (Female),te,female
8,kn_male,Kannada (Male),kn,male
9,kn_female,Kannada (Female),kn,female


## 3. List Available Models
OpenAI-compatible `GET /v1/models` — returns the TTS models supported by this server.

In [21]:
try:
    response = requests.get(f"{BASE_URL}/v1/models", timeout=5)
    response.raise_for_status()
    models_data = response.json()

    print(f"🤖 Available TTS Models ({len(models_data['data'])}):\n")
    df_models = pd.DataFrame(models_data["data"])
    display(df_models[["id", "owned_by", "created"]])

except Exception as e:
    print(f"❌ Failed to list models: {e}")

🤖 Available TTS Models (2):



,id,owned_by,created
0,tts-1,svara,1677610602
1,tts-1-hd,svara,1677610602


## 4. Generate Speech — Hindi Male (`onyx` voice)
Uses the OpenAI-compatible `POST /v1/audio/speech` endpoint.  
Voice mapping: `onyx` → `hi_male`

In [ ]:
# OpenAI-compatible request body  (POST /v1/audio/speech)
payload = {
    "model": "tts-1",
    "input": "नमस्ते, मैं स्वर हूँ।",   # Hindi greeting
    "voice": "onyx",                       # maps to hi_male
    "response_format": "wav",
    "speed": 1.0,
}

print("🎤 Synthesizing Hindi Male speech via /v1/audio/speech ...")
print("⏱️  Note: CPU inference is slow (~1-2 tokens/sec). This may take a few minutes.")

try:
    response = requests.post(
        f"{BASE_URL}/v1/audio/speech",
        json=payload,
        timeout=600,
    )
    response.raise_for_status()

    audio_bytes = response.content
    print(f"✅ Audio generated | size={len(audio_bytes):,} bytes | content-type={response.headers.get('content-type')}")

    # Inline playback
    display(Audio(audio_bytes, autoplay=False, rate=24000))

    # Persist to disk
    output_path = "output_hindi_male.wav"
    with open(output_path, "wb") as f:
        f.write(audio_bytes)
    print(f"💾 Saved → {output_path}")

except requests.HTTPError as e:
    print(f"❌ HTTP {e.response.status_code}: {e.response.text}")
except Exception as e:
    print(f"❌ Request failed: {e}")

🎤 Synthesizing Hindi Male speech via /v1/audio/speech ...
⏱️  Note: CPU inference is slow (~1-2 tokens/sec). This may take a few minutes.


## 5. Generate Speech — English Female (`nova` voice)
Uses the OpenAI-compatible `POST /v1/audio/speech` endpoint.  
Voice mapping: `nova` → `en_female`

In [19]:
# OpenAI-compatible request body  (POST /v1/audio/speech)
payload = {
    "model": "tts-1",
    "input": "Hello, I am Svara — an enterprise-grade text to speech system.",
    "voice": "nova",             # maps to en_female
    "response_format": "wav",
    "speed": 1.0,
}

print("🎤 Synthesizing English Female speech via /v1/audio/speech ...")
print("⏱️  Note: CPU inference is slow (~1-2 tokens/sec). This may take a few minutes.")

try:
    response = requests.post(
        f"{BASE_URL}/v1/audio/speech",
        json=payload,
        timeout=600,
    )
    response.raise_for_status()

    audio_bytes = response.content
    print(f"✅ Audio generated | size={len(audio_bytes):,} bytes | content-type={response.headers.get('content-type')}")

    # Inline playback
    display(Audio(audio_bytes, autoplay=False, rate=24000))

    # Persist to disk
    output_path = "output_english_female.wav"
    with open(output_path, "wb") as f:
        f.write(audio_bytes)
    print(f"💾 Saved → {output_path}")

except requests.HTTPError as e:
    print(f"❌ HTTP {e.response.status_code}: {e.response.text}")
except Exception as e:
    print(f"❌ Request failed: {e}")

🎤 Synthesizing English Female speech via /v1/audio/speech ...
⏱️  Note: CPU inference is slow (~1-2 tokens/sec). This may take a few minutes.
❌ HTTP 404: {"detail":"Not Found"}


## 6. Custom Voice Test — OpenAI API
Choose any OpenAI voice alias and provide your own text.

| OpenAI Voice | Svara Voice  | Language       |
|:-------------|:-------------|:---------------|
| `alloy`      | `en_male`    | English Male   |
| `echo`       | `en_male`    | English Male   |
| `fable`      | `en_female`  | English Female |
| `onyx`       | `hi_male`    | Hindi Male     |
| `nova`       | `en_female`  | English Female |
| `shimmer`    | `hi_female`  | Hindi Female   |

In [20]:
# ── Configure your request here ─────────────────────────────────────────────
CUSTOM_TEXT  = "Welcome to Svara, the enterprise text-to-speech engine."
CUSTOM_VOICE = "alloy"   # alloy | echo | fable | onyx | nova | shimmer
CUSTOM_MODEL = "tts-1"   # tts-1 | tts-1-hd
OUTPUT_FILE  = "output_custom.wav"
# ─────────────────────────────────────────────────────────────────────────────

payload = {
    "model": CUSTOM_MODEL,
    "input": CUSTOM_TEXT,
    "voice": CUSTOM_VOICE,
    "response_format": "wav",
    "speed": 1.0,
}

print(f"🎤 Custom TTS | model={CUSTOM_MODEL} voice={CUSTOM_VOICE}")
print(f"   Text: \"{CUSTOM_TEXT[:80]}{'...' if len(CUSTOM_TEXT) > 80 else ''}\"")
print("⏱️  CPU inference — please wait ...")

try:
    response = requests.post(
        f"{BASE_URL}/v1/audio/speech",
        json=payload,
        timeout=600,
    )
    response.raise_for_status()

    audio_bytes = response.content
    print(f"\n✅ Done | size={len(audio_bytes):,} bytes")

    display(Audio(audio_bytes, autoplay=True, rate=24000))

    with open(OUTPUT_FILE, "wb") as f:
        f.write(audio_bytes)
    print(f"💾 Saved → {OUTPUT_FILE}")

except requests.HTTPError as e:
    print(f"❌ HTTP {e.response.status_code}: {e.response.text}")
except Exception as e:
    print(f"❌ Request failed: {e}")

🎤 Custom TTS | model=tts-1 voice=alloy
   Text: "Welcome to Svara, the enterprise text-to-speech engine."
⏱️  CPU inference — please wait ...
❌ HTTP 404: {"detail":"Not Found"}
